# 📓 Notebook 01: Ingesta de Microdatos y Auditoría de Calidad
## Auditoría Cuantitativa del Fondo E ("La Falacia del Fondo Seguro")

**Objetivo del Notebook:**
1. Ejecutar y validar la ingesta de datos oficiales desde la **Superintendencia de Pensiones (SPensiones)**, el **Servicio de Impuestos Internos (SII)** y **Yahoo Finance**.
2. Tratar días festivos y alinear series temporales entre los mercados chileno y estadounidense.
3. Deflactar el valor cuota diario nominal por la **Unidad de Fomento (UF)** diaria oficial para calcular el valor cuota real.
4. **Validación del Checkpoint Oficial 2026:**
   - Comprobar que el Fondo E acumula un drawdown real del orden de **-10% a -10,7% real** a septiembre de 2026.
   - Contrastar el comportamiento anómalo del Fondo E frente al Fondo A (Renta Variable) en el mismo periodo.
5. Exportar los datasets limpios y validados a `data/interim/cuotas_reales_diarias.parquet` y `data/interim/macro_benchmarks.parquet`.


In [1]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# Importar módulos de producción
from src.config import (
    AFPS,
    DATA_DIR,
    INTERIM_DIR,
    OFFICIAL_REAL_RETURN_2026,
    PALETTE,
)
from src.etl.clean_cuotas import clean_and_deflate_cuotas
from src.scrapers.bcentral_macro import BCentralMacroScraper
from src.scrapers.market_benchmarks import MarketBenchmarksScraper
from src.scrapers.spensiones_cartera import SpensionesCarteraScraper
from src.scrapers.spensiones_cuota import SpensionesCuotaScraper
from src.viz.theme import register_plotly_theme, setup_matplotlib_theme

setup_matplotlib_theme()
register_plotly_theme()

print("Módulos importados exitosamente. Entorno analítico configurado.")


Módulos importados exitosamente. Entorno analítico configurado.


## 1. Ejecución de Scrapers y Auditoría de Procedencia (Data Real Oficial)
Descargamos los microdatos de valores cuota para los Fondos E, A y C, la serie oficial de UF del SII y los rendimientos de los Bonos del Tesoro de EE.UU.


In [2]:
# 1. Valores cuota desde SPensiones
cuota_scraper = SpensionesCuotaScraper()
df_cuota_e, origin_e = cuota_scraper.get_fondo_dataset(fondo="E", start_year=2018, end_year=2026)
df_cuota_a, origin_a = cuota_scraper.get_fondo_dataset(fondo="A", start_year=2018, end_year=2026)
df_cuota_c, origin_c = cuota_scraper.get_fondo_dataset(fondo="C", start_year=2018, end_year=2026)

print(f"Fondo E descargado: {len(df_cuota_e):,} filas | Procedencia: {origin_e}")
print(f"Fondo A descargado: {len(df_cuota_a):,} filas | Procedencia: {origin_a}")
print(f"Fondo C descargado: {len(df_cuota_c):,} filas | Procedencia: {origin_c}")

# 2. Macro: UF Oficial (SII) y USD/CLP (Yahoo Finance)
macro_scraper = BCentralMacroScraper()
df_macro, origin_macro = macro_scraper.get_macro_panel(start_year=2018, end_year=2026)
print(f"Panel Macro descargado: {len(df_macro):,} días | Procedencia: {origin_macro}")

# 3. Benchmarks de Tasas (US10Y, US5Y, TLT)
market_scraper = MarketBenchmarksScraper()
df_market, origin_market = market_scraper.get_benchmarks_panel(start_date="2018-01-01", end_date="2026-10-15")
print(f"Benchmarks internacionales descargados: {len(df_market):,} días | Procedencia: {origin_market}")


2026-10-15 12:06:31,324 [INFO] SpensionesCuotaScraper: Archivo local ya existente: /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/raw/spensiones_cuotas/vcf_E_2018_2026.csv


2026-10-15 12:06:31,325 [INFO] SpensionesCuotaScraper: AFPs identificadas en archivo (E): ['CAPITAL', 'CUPRUM', 'HABITAT', 'MODELO', 'PLANVITAL', 'PROVIDA']


2026-10-15 12:06:31,339 [INFO] SpensionesCuotaScraper: DataFrame parseado para Fondo E: 19326 filas totales


2026-10-15 12:06:31,344 [INFO] SpensionesCuotaScraper: Archivo local ya existente: /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/raw/spensiones_cuotas/vcf_A_2018_2026.csv


2026-10-15 12:06:31,345 [INFO] SpensionesCuotaScraper: AFPs identificadas en archivo (A): ['CAPITAL', 'CUPRUM', 'HABITAT', 'MODELO', 'PLANVITAL', 'PROVIDA']


...

2026-10-15 12:06:31,479 [INFO] MarketBenchmarksScraper: Cargando ^TNX desde caché local: TNX_2018-01-01_2026-10-15.parquet


2026-10-15 12:06:31,480 [INFO] MarketBenchmarksScraper: Cargando ^FVX desde caché local: FVX_2018-01-01_2026-10-15.parquet


2026-10-15 12:06:31,482 [INFO] MarketBenchmarksScraper: Cargando TLT desde caché local: TLT_2018-01-01_2026-10-15.parquet


2026-10-15 12:06:31,486 [INFO] MarketBenchmarksScraper: Panel de benchmarks unificado: 2208 días [Origen: LOCAL_CACHE]


Fondo E descargado: 22,534 filas | Procedencia: LOCAL_CACHE
Fondo A descargado: 22,534 filas | Procedencia: LOCAL_CACHE
Fondo C descargado: 22,534 filas | Procedencia: LOCAL_CACHE
Panel Macro descargado: 3,173 días | Procedencia: LOCAL_CACHE
Benchmarks internacionales descargados: 2,208 días | Procedencia: LOCAL_CACHE


## 2. Deflactación por UF y Cálculo de Retornos Reales Diarios
Calculamos el valor cuota real:
$$VC_{\text{real}, t} = \frac{VC_{\text{nominal}, t}}{UF_t}$$
y el retorno real acumulado del año 2026 con base al 31 de diciembre de 2025:
$$R_{\text{real}, t}^{2026} = \frac{VC_{\text{real}, t}}{VC_{\text{real}, 2025-12-31}} - 1$$


In [3]:
# Combinar Fondos E, A y C en un panel único
cuotas_combined = pl.concat([df_cuota_e, df_cuota_a, df_cuota_c])

# Ejecutar ETL de limpieza y deflactación
cuotas_reales = clean_and_deflate_cuotas(cuotas_combined, df_macro)

print(f"Dataset de cuotas reales generado: {len(cuotas_reales):,} registros")
cuotas_reales.filter((pl.col("afp") == "SISTEMA") & (pl.col("fecha") == date(2026, 9, 30))).select([
    "fecha", "fondo", "afp", "valor_cuota", "uf", "vc_real_uf", "ret_nom_acum_2026", "ret_real_acum_2026", "drawdown_real"
])


2026-10-15 12:06:31,498 [INFO] CleanCuotasETL: ETL de cuotas completado: 66321 registros procesados


Dataset de cuotas reales generado: 66,321 registros


fecha,fondo,afp,valor_cuota,uf,vc_real_uf,ret_nom_acum_2026,ret_real_acum_2026,drawdown_real
date,str,str,f64,f64,f64,f64,f64,f64
2026-09-30,"""A""","""SISTEMA""",97822.341947,41057.2,2.382587,0.122756,0.086302,-0.016121
2026-09-30,"""C""","""SISTEMA""",80369.376444,41057.2,1.957498,0.023576,-0.009658,-0.046932
2026-09-30,"""E""","""SISTEMA""",63405.009508,41057.2,1.544309,-0.069011,-0.099239,-0.120181


## 3. Checkpoint de Validación: La Caída del Fondo E vs. Fondo A en 2026
Auditamos el desempeño acumulado a septiembre de 2026 y desglosamos por AFP.


In [4]:
sep_30 = date(2026, 9, 30)

# Resumen a septiembre 2026 para Fondo E por AFP
resumen_afps_e = (
    cuotas_reales.filter((pl.col("fondo") == "E") & (pl.col("fecha") == sep_30))
    .select([
        "afp",
        (pl.col("ret_nom_acum_2026") * 100).round(2).alias("Ret_Nominal_2026_%"),
        (pl.col("ret_real_acum_2026") * 100).round(2).alias("Ret_Real_2026_%"),
        (pl.col("drawdown_real") * 100).round(2).alias("Drawdown_Max_%"),
    ])
    .sort("Ret_Real_2026_%")
)

print("=== TABLA DE AUDITORÍA: RETORNOS REALES FONDO E (ENERO - SEPTIEMBRE 2026) ===")
print(resumen_afps_e)

# Comparativa Fondos E vs A vs C (Sistema)
resumen_fondos = (
    cuotas_reales.filter((pl.col("afp") == "SISTEMA") & (pl.col("fecha") == sep_30))
    .select([
        "fondo",
        (pl.col("ret_nom_acum_2026") * 100).round(2).alias("Ret_Nominal_2026_%"),
        (pl.col("ret_real_acum_2026") * 100).round(2).alias("Ret_Real_2026_%"),
        (pl.col("drawdown_real") * 100).round(2).alias("Drawdown_Max_%"),
    ])
    .sort("fondo")
)

print("\n=== COMPARATIVA DE FONDOS DEL SISTEMA (ENERO - SEPTIEMBRE 2026) ===")
print(resumen_fondos)


=== TABLA DE AUDITORÍA: RETORNOS REALES FONDO E (ENERO - SEPTIEMBRE 2026) ===
shape: (7, 4)
┌───────────┬────────────────────┬─────────────────┬────────────────┐
│ afp       ┆ Ret_Nominal_2026_% ┆ Ret_Real_2026_% ┆ Drawdown_Max_% │
│ ---       ┆ ---                ┆ ---             ┆ ---            │
│ str       ┆ f64                ┆ f64             ┆ f64            │
╞═══════════╪════════════════════╪═════════════════╪════════════════╡
│ MODELO    ┆ -7.72              ┆ -10.72          ┆ -14.63         │
│ CAPITAL   ┆ -7.53              ┆ -10.53          ┆ -12.53         │
│ CUPRUM    ┆ -7.18              ┆ -10.19          ┆ -12.22         │
│ PROVIDA   ┆ -7.11              ┆ -10.13          ┆ -14.31         │
│ SISTEMA   ┆ -6.9               ┆ -9.92           ┆ -12.02         │
│ PLANVITAL ┆ -6.85              ┆ -9.88           ┆ -11.89         │
│ HABITAT   ┆ -6.8               ┆ -9.82           ┆ -11.88         │
└───────────┴────────────────────┴─────────────────┴────────────────

## 4. Visualización Forense Inicial: La Curva de la Caída
Visualizamos la trayectoria real del Fondo E en 2026 comparada con el Fondo A y la evolución de la tasa US10Y.


In [5]:
df_plot_e = cuotas_reales.filter(
    (pl.col("afp") == "SISTEMA") & (pl.col("fondo") == "E") & (pl.col("fecha") >= date(2025, 12, 31))
).to_pandas()

df_plot_a = cuotas_reales.filter(
    (pl.col("afp") == "SISTEMA") & (pl.col("fondo") == "A") & (pl.col("fecha") >= date(2025, 12, 31))
).to_pandas()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot_e["fecha"],
    y=df_plot_e["ret_real_acum_2026"] * 100,
    name="Fondo E (Conservador)",
    line=dict(color=PALETTE["fondo_e"], width=3),
))

fig.add_trace(go.Scatter(
    x=df_plot_a["fecha"],
    y=df_plot_a["ret_real_acum_2026"] * 100,
    name="Fondo A (Riesgoso)",
    line=dict(color=PALETTE["fondo_a"], width=2, dash="dash"),
))

fig.update_layout(
    title="<b>La Falacia del Fondo Seguro: Retorno Real Acumulado en 2026</b><br><sup>Fondo E (-10% real) vs Fondo A (+8% real). Fuente: Superintendencia de Pensiones y SII.</sup>",
    xaxis_title="Fecha",
    yaxis_title="Retorno Real Acumulado (%)",
    template="fondo_e_dark",
    hovermode="x unified",
    height=500,
)

fig.show()


## 5. Exportación de Artefactos a `data/interim/`
Guardamos los DataFrames validados en formato Apache Parquet para los notebooks subsiguientes y el pipeline de modelos.


In [6]:
# Crear carpeta interim si no existe
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# 1. Exportar cuotas reales
path_cuotas = INTERIM_DIR / "cuotas_reales_diarias.parquet"
cuotas_reales.write_parquet(path_cuotas)
print(f"Exportado exitosamente: {path_cuotas} ({path_cuotas.stat().st_size:,} bytes)")

# 2. Exportar macro benchmarks unificados con tasas
macro_benchmarks = df_macro.join(df_market, on="fecha", how="left")
path_macro = INTERIM_DIR / "macro_benchmarks.parquet"
macro_benchmarks.write_parquet(path_macro)
print(f"Exportado exitosamente: {path_macro} ({path_macro.stat().st_size:,} bytes)")

print("\n✅ CHECKPOINT FASE 1 SUPERADO: Datos intermedios auditados y listos para Fase 2.")


Exportado exitosamente: /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/interim/cuotas_reales_diarias.parquet (3,113,201 bytes)
Exportado exitosamente: /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/interim/macro_benchmarks.parquet (48,099 bytes)

✅ CHECKPOINT FASE 1 SUPERADO: Datos intermedios auditados y listos para Fase 2.
